# Gemini + VecDB RAG
Deliver retrieval-augmented answers by pairing Oracle VecDB's vector search with Google Gemini's multimodal reasoning.


## 1. Scenario Overview
This notebook walks through an end-to-end VecDB + Gemini workflow:
1. Load shared `.env` credentials to configure both Oracle VecDB and the Gemini SDK.
2. Chunk sample documents, generate embeddings with Gemini, and persist vectors/metadata in VecDB.
3. Execute semantic search queries against VecDB to gather relevant passages and context.
4. Prompt Gemini with the retrieved snippets to produce grounded, citeable responses.
5. Tear down demo artifacts so the environment stays clean for future runs.


## 2. Setup
Install the required SDKs and initialize environment-driven configuration for Oracle VecDB and Gemini. The notebook prints all resolved settings so users can confirm model names and connection targets before proceeding.


In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas google-generativeai


In [ ]:
import os
import textwrap
from dataclasses import dataclass
from typing import List

from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration
import google.generativeai as genai

print('Loading environment variables for VecDB and Gemini...')
load_dotenv()

vecdb_host = os.getenv('VECDB_REST_URL')
vecdb_user = os.getenv('VECDB_USERNAME') or os.getenv('VECDB_USER')
vecdb_password = os.getenv('VECDB_PASSWORD')
vecdb_access_token = os.getenv('VECDB_ACCESS_TOKEN')
gemini_key = os.getenv('GOOGLE_API_KEY')
embedding_model = os.getenv('GEMINI_EMBED_MODEL', 'text-embedding-004')
chat_model_name = os.getenv('GEMINI_CHAT_MODEL') or 'gemini-3.1-flash-lite'
if chat_model_name in {'gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-2.5-pro'}:
    chat_model_name = 'gemini-3.1-flash-lite'
table_name = os.getenv('GEMINI_DOC_TABLE', 'GEMINI_DOCS_DEMO')

print(f'VecDB host: {vecdb_host}')
print(f'VecDB user: {vecdb_user}')
print(f'Gemini embed model: {embedding_model}')
print(f'Gemini chat model: {chat_model_name}')

vecdb_config_kwargs = {"rest_url": vecdb_host}
if vecdb_access_token:
    vecdb_config_kwargs["access_token"] = vecdb_access_token
else:
    vecdb_config_kwargs["username"] = vecdb_user
    vecdb_config_kwargs["password"] = vecdb_password
vecdb_config = Configuration(**vecdb_config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    vecdb_config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification (self-signed certificates).')

vecdb = OracleVecDB(vecdb_config)
auth_method = 'bearer token' if vecdb_config.access_token else 'username/password'
print('Connected to VecDB:', vecdb_config.rest_url)
print('Auth method:', auth_method)

if not gemini_key:
    raise RuntimeError('Missing GOOGLE_API_KEY in environment. Set it before running Gemini cells.')

genai.configure(api_key=gemini_key)
print('Configured Gemini client.')


## 3. Prepare Sample Documents
Create a lightweight corpus that mirrors real VecDB workloads. Each document includes prose plus metadata we will preserve for downstream citation and governance examples.


In [ ]:
@dataclass
class Document:
    id: str
    title: str
    text: str

DOCS = [
    Document(
        'gem-doc-1',
        'VecDB developer overview',
        textwrap.dedent('''
            Oracle VecDB provides a managed vector database service designed for enterprise workloads.
            It supports hybrid vector + metadata filtering, automatic indexing, and tight integration with Oracle Autonomous Database.
            This overview describes how teams can onboard datasets, configure collections, and monitor usage.
        ''').strip(),
    ),
    Document(
        'gem-doc-2',
        'Gemini model family',
        textwrap.dedent('''
            Gemini models unify multimodal reasoning across text, code, and audio.
            The flash variants are optimized for fast chat, while the pro tiers boost reasoning depth.
            Developers can call the embeddings API to convert content to high-quality vector representations.
        ''').strip(),
    ),
    Document(
        'gem-doc-3',
        'RAG design tips',
        textwrap.dedent('''
            Retrieval-augmented generation combines a retrieval step with an LLM response.
            Key design points include chunking documents, capturing metadata, and grounding prompts with citations.
            Cleanup routines keep demo environments ready for repetitive testing.
        ''').strip(),
    ),
]

print(f'Loaded {len(DOCS)} source documents.')


## 4. Embed and Upsert
Use Gemini to generate dense embeddings for each chunk, then upsert the vectors—and descriptive metadata—into Oracle VecDB. This section also highlights how VecDB handles table lifecycle management.


In [ ]:
import pandas as pd
from uuid import uuid4

print('Preparing VecDB table...')
try:
    vecdb.drop_vector_table(name=table_name)
    print('Dropped existing table', table_name)
except Exception as exc:
    print('Drop table skipped')

vecdb.create_vector_table(
    name=table_name,
    comment='Gemini RAG demo',
    annotations={'TITLE': 'string', 'SOURCE': 'string', 'DOC_ID': 'string', 'TEXT': 'string'},
)
print('Created table', table_name)

def chunk_document(doc: Document, max_chars: int = 400) -> List[dict]:
    chunks = []
    words = doc.text.split()
    buffer = []
    current = 0
    for word in words:
        next_len = len(word) + (1 if buffer else 0)
        if buffer and current + next_len > max_chars:
            chunks.append({'content': ' '.join(buffer)})
            buffer = []
            current = 0
        buffer.append(word)
        current += next_len
    if buffer:
        chunks.append({'content': ' '.join(buffer)})
    return chunks

records = []
for doc in DOCS:
    for chunk in chunk_document(doc):
        records.append({
            'doc_id': doc.id,
            'title': doc.title,
            'text': chunk['content'],
            'source': 'gemini-demo'
        })

print('Generated', len(records), 'chunks total.')

def embed_texts(texts: List[str]) -> List[List[float]]:
    vectors = []
    for text in texts:
        response = genai.embed_content(
            model=embedding_model,
            content=text,
        )
        vectors.append(response['embedding'])
    return vectors

vectors = embed_texts([row['text'] for row in records])

rows = []
for record, vector in zip(records, vectors):
    rows.append({
        'id': str(uuid4()),
        'dense_vector': vector,
        'metadata': {
            'TITLE': record['title'],
            'SOURCE': record['source'],
            'DOC_ID': record['doc_id'],
            'TEXT': record['text'],
        }
    })

vecdb.upsert_vectors(table_name=table_name, vectors=rows)
print('Upserted', len(rows), 'chunks into VecDB')

pd.DataFrame(records).head()


In [ ]:
print('Table description:')
vecdb.describe_vector_table(name=table_name)


## 5. Retrieve Context from VecDB
Run a semantic query against VecDB to fetch top matching passages. Printed results include titles, document IDs, and truncated snippets so analysts can inspect the retrieved context.


In [ ]:
def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)


def result_vector(item):
    if isinstance(item, dict):
        return item.get("vector") or item.get("dense_vector")
    return getattr(item, "vector", getattr(item, "dense_vector", None))


def result_text(item):
    return item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")


query_text = 'How do Gemini embeddings support VecDB RAG workflows?'
print('Searching VecDB for:', query_text)
results = vecdb.query(
    table_name=table_name,
    query_by={
        'vector': embed_texts([query_text])[0],
        'vector_field': 'dense_vector'
    },
    top_k=5,
    include_vectors=False
)
items = query_items(results)
print('VecDB results:', len(items))
for idx, item in enumerate(items, start=1):
    metadata = result_metadata(item)
    snippet = metadata.get('TEXT') or result_text(item) or ''
    normalized = ' '.join(snippet.split())
    print()
    print(f'Result {idx}')
    print(f"  TITLE: {metadata.get('TITLE')}")
    print(f"  DOC_ID: {metadata.get('DOC_ID')}")
    print(f"  TEXT: {normalized}")


## 6. Prompt Gemini with Retrieved Context
Feed the VecDB search results into Gemini to answer domain questions while citing specific sources. This illustrates how Gemini can remain grounded when VecDB supplies curated evidence.


In [ ]:
def build_prompt(question: str, hits: list) -> str:
    snippets = []
    for item in query_items(hits):
        meta = result_metadata(item)
        snippet = meta.get('TEXT') or result_text(item) or ''
        title = meta.get('TITLE', 'Untitled')
        doc_id = meta.get('DOC_ID', 'N/A')
        snippets.append(f"Source: {title} (DOC_ID={doc_id})\n{snippet}")
    joined = "\n\n".join(snippets)
    instructions = (
        "You are helping explain how to combine Oracle VecDB with Google Gemini.\n"
        "Provide a concise answer to the question using the context. Cite sources when relevant."
    )
    return f"{instructions}\n\nQuestion: {question}\n\nContext:\n{joined}\n\nAnswer:\n"

prompt = build_prompt(query_text, results)
model = genai.GenerativeModel(chat_model_name)
gemini_reply = model.generate_content(prompt)
print("\nGemini response:\n")
print(textwrap.fill(gemini_reply.text, width=150))

## 7. Optional: Interactive Q&A Helper
Wrap retrieval and prompting in a helper so users can iterate on follow-up questions without repeating boilerplate code.


In [ ]:
def answer_question(question: str, k: int = 5) -> str:
    hits = vecdb.query(
        table_name=table_name,
        query_by={
            'vector': embed_texts([question])[0],
            'vector_field': 'dense_vector'
        },
        top_k=k,
        include_vectors=False
    )
    prompt = build_prompt(question, hits)
    response = model.generate_content(prompt)
    return textwrap.fill(response.text, width=150)

response_text = answer_question('What best practices keep VecDB RAG grounded?')
print('Gemini response:')
print(response_text)


## 8. Cleanup
Drop the demo table to keep VecDB tidy. This mirrors the hygiene practices recommended across the VizDB demo suite.


In [ ]:
print('Dropping table:', table_name)
try:
    vecdb.drop_vector_table(name=table_name)
    print('Dropped table.')
except Exception as exc:
    print('drop_vector_table error:', exc)
